# 1. Setting Up
This is to initialize all necessary requirements and dataset used for the DATAMIN Finals project.

In [ ]:
# Run this if you don't have the libraries installed
# This was added for those that are running the notebook 
# in a new environment and don't have the libraries installed yet. 
# You can skip this out if you already have them installed.

%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install scikit-learn


In [ ]:
# This imports all the libraries we need for this notebook.
import numpy as np
import pandas as pd

# This is for plotting graphs and visualizations.
import matplotlib.pyplot as plt

# Prepare features for clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# This is just to make sure that when we print out dataframes, 
# we can see all the columns and the full width of the dataframe.
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)


In [ ]:
# This reads in the data from the CSV file and gives us a quick 
# look at the first 10 rows. The info about the dataframe, 
# and checks for any missing values.
# This is optional.

df = pd.read_csv("./data/IBP_OBS_Data_Launch2-OBS_Data_2006-2025.csv")

display(df.head(5))

In [ ]:
## Simple check to see what the data looks like, and if there are any missing values.

df.info()
df.isna().sum()

# 2. Selecting Necessary Data
After all the setup, the next step is to filter out all unecessary data. This is to limit all the information that will be presented to only ones that are necessary for analysis.

In [ ]:
# Select only the columns needed for analysis
# Keeping identifiers, core scores, and comparability flags

selected_columns = [
    'country',           # Country name
    'ISO',               # Country code
    'year',              # Survey year
    'region',            # Broad region
    'comp_region',       # Detailed region
    'obi',               # Transparency score (0-100)
    'oversight_obi',     # Oversight score (0-100)
    'pub_eng',           # Public participation score (0-100)
    'comp1215',          # Comparability from 2012 onward
    'comp1517',          # Comparability from 2015 onward
    'comp1719',          # Comparability from 2017 onward
    'comp1921',          # Comparability from 2019 onward
    'comp2123',          # Comparability from 2021 onward
    'comp2325'           # Comparability from 2023 onward
]

# Create a new dataframe with only the selected columns
df_selected = df[selected_columns].copy()

# Display first few rows to verify
print("Selected columns shape:", df_selected.shape)
display(df_selected.head())

# 3. Data Cleaning & Saving
Normalizing all data variables (if given dataset is dirty)

*Dirty data just means data that may have appeared multiple times in different forms Ex. Male, M, m, male*<br/>
*Null values are also dirty data due to how it should be handled when analyzing given data*<br/>
*Removal of duplicate entries and many more also also considered ion this step*

Save the cleaned dataset when it has been fully adjusted.

*This step is ONLY necessary if the given dataset is not normalized*

In [ ]:
# Convert numeric columns from string/object to proper numeric types
# This ensures calculations work correctly

numeric_columns = ['obi', 'oversight_obi', 'pub_eng']

for col in numeric_columns:
    df_selected[col] = pd.to_numeric(df_selected[col], errors='coerce')

# Check for missing values in key columns
print("Missing values in key columns:")
print(df_selected[numeric_columns].isna().sum())

# Optional: Drop rows where obi is missing (if needed for analysis)
# df_clean = df_selected.dropna(subset=['obi'])

# Save the cleaned dataset to a new CSV file
df_selected.to_csv('./data/IBP_OBS_cleaned.csv', index=False)
print("Cleaned dataset saved to './data/IBP_OBS_cleaned.csv'")

display(df_selected.head())

# Quick summary statistics
print("\nSummary statistics for key scores (2006-2025):")
print(df_selected[numeric_columns].describe())

# 4. Data Analysis

In [ ]:
# Calculate global average OBI score per year
# Exclude the problematic '2023_pilot' year

# Filter out pilot year
df_trend = df_selected[df_selected['year'] != '2023_pilot'].copy()

# Convert year to numeric if needed
df_trend['year'] = pd.to_numeric(df_trend['year'])

# Calculate yearly average FROM THE FILTERED DATA
yearly_avg = df_trend.groupby('year')['obi'].mean().reset_index()

print("Global average OBI score by year (excluding pilot):")
print(yearly_avg)

# Create the plot
plt.figure(figsize=(10, 6))
plt.plot(yearly_avg['year'], yearly_avg['obi'], marker='o', linewidth=2, markersize=8)

# Set x-axis to show only the years that exist in the data
plt.xticks(yearly_avg['year'])  # This ensures only 2006, 2008, 2010, etc. show

plt.title('Global Average Open Budget Index (OBI) Score Over Time')
plt.xlabel('Year')
plt.ylabel('Average OBI Score (0-100)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Analysis 2: Regional Comparison (2025)
# Which regions are most transparent?

# Filter to 2025 data
df_2025 = df_selected[df_selected['year'] == 2025].copy()

# Calculate average OBI by region
region_avg = df_2025.groupby('region')['obi'].mean().sort_values(ascending=False)

print("=" * 50)
print("REGIONAL ANALYSIS (2025)")
print("=" * 50)
print("\nAverage OBI score by region:")
print(region_avg.round(1))

# Bar plot
plt.figure(figsize=(10, 6))
region_avg.plot(kind='bar', color='steelblue')
plt.axhline(y=61, color='r', linestyle='--', linewidth=1.5, label='Sufficient Threshold (61)')
plt.title('Average Transparency Score by Region (2025)')
plt.xlabel('Region')
plt.ylabel('Average OBI Score (0-100)')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Analysis 3: Pillar Correlations
# Does transparency predict oversight?

# Focus on 2025 where all pillars are available
df_pillars = df_selected[df_selected['year'] == 2025].dropna(subset=['obi', 'oversight_obi'])

print("=" * 50)
print("PILLAR CORRELATION ANALYSIS (2025)")
print("=" * 50)

# Calculate correlation
pearson_corr = df_pillars['obi'].corr(df_pillars['oversight_obi'])
print(f"\nPearson correlation (Transparency vs Oversight): {pearson_corr:.3f}")

# Scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(df_pillars['obi'], df_pillars['oversight_obi'], alpha=0.7, s=80, color='green')

# Add trend line
z = np.polyfit(df_pillars['obi'], df_pillars['oversight_obi'], 1)
p = np.poly1d(z)
plt.plot(df_pillars['obi'], p(df_pillars['obi']), 'r--', linewidth=2, label=f'Trend line (r={pearson_corr:.2f})')

plt.xlabel('Transparency Score (OBI)')
plt.ylabel('Oversight Score')
plt.title('Transparency vs. Oversight (2025)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Analysis 4: Country-Level Changes
# Which countries improved or declined the most?

# Calculate change from earliest to latest year per country
first_obs = df_selected.groupby('country')['year'].min().reset_index()
last_obs = df_selected.groupby('country')['year'].max().reset_index()

first_scores = first_obs.merge(df_selected[['country', 'year', 'obi']], on=['country', 'year'])
last_scores = last_obs.merge(df_selected[['country', 'year', 'obi']], on=['country', 'year'])

change_df = first_scores[['country']].copy()
change_df['first_year'] = first_scores['year']
change_df['first_obi'] = first_scores['obi']
change_df['last_year'] = last_scores['year']
change_df['last_obi'] = last_scores['obi']
change_df['change'] = change_df['last_obi'] - change_df['first_obi']
change_df = change_df.dropna()

print("=" * 50)
print("COUNTRY-LEVEL CHANGES")
print("=" * 50)

print("\nTop 5 Improvers (most positive change):")
print(change_df.nlargest(5, 'change')[['country', 'first_obi', 'last_obi', 'change']].round(1))

print("\nTop 5 Decliners (most negative change):")
print(change_df.nsmallest(5, 'change')[['country', 'first_obi', 'last_obi', 'change']].round(1))

In [ ]:
# Analysis 5: The 2021-2023 Drop
# Which countries drove the global decline?

df_2021 = df_selected[df_selected['year'] == 2021][['country', 'obi']]
df_2023 = df_selected[df_selected['year'] == 2023][['country', 'obi']]

change_2021_2023 = df_2021.merge(df_2023, on='country', suffixes=('_2021', '_2023'))
change_2021_2023['change'] = change_2021_2023['obi_2023'] - change_2021_2023['obi_2021']

# Global change
global_change = change_2021_2023['change'].mean()
print("=" * 50)
print("2021-2023 DROP INVESTIGATION")
print("=" * 50)
print(f"\nGlobal average change from 2021 to 2023: {global_change:.2f} points")

print("\nCountries with largest drop (2021 → 2023):")
print(change_2021_2023.nsmallest(5, 'change')[['country', 'obi_2021', 'obi_2023', 'change']].round(1))

print("\nCountries with largest improvement (2021 → 2023):")
print(change_2021_2023.nlargest(5, 'change')[['country', 'obi_2021', 'obi_2023', 'change']].round(1))

# Visualize the distribution of changes
plt.figure(figsize=(10, 6))
plt.hist(change_2021_2023['change'].dropna(), bins=15, color='skyblue', edgecolor='black')
plt.axvline(x=0, color='r', linestyle='--', linewidth=1.5, label='No change')
plt.axvline(x=global_change, color='darkred', linestyle='-', linewidth=2, label=f'Global average ({global_change:.2f})')
plt.xlabel('Change in OBI Score (2021 to 2023)')
plt.ylabel('Number of Countries')
plt.title('Distribution of Country-Level Changes (2021-2023)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Summary Statistics for your PPT
print("=" * 50)
print("SUMMARY STATISTICS")
print("=" * 50)

print(f"\nTotal countries in dataset: {df_selected['country'].nunique()}")
print(f"Years covered: {sorted(df_selected['year'].unique())}")
print(f"\nGlobal OBI range: {df_selected['obi'].min():.1f} to {df_selected['obi'].max():.1f}")
print(f"Global OBI average (all years): {df_selected['obi'].mean():.1f}")

# Best and worst performing countries in 2025
df_2025 = df_selected[df_selected['year'] == 2025]
best = df_2025.loc[df_2025['obi'].idxmax()]
worst = df_2025.loc[df_2025['obi'].idxmin()]

print(f"\nHighest transparency (2025): {best['country']} ({best['obi']:.1f})")
print(f"Lowest transparency (2025): {worst['country']} ({worst['obi']:.1f})")